In [ ]:
// onClass\server.js
const { generateText } = require("ai");

const result = await generateText({
    model: "anthropic/claude-haiku-4.5",
    prompt: "Xin chào!",
});

console.log(result);


In [ ]:
{
    "dependencies": {
        "ai": "^6.0.116",
        "dotenv": "^17.3.1"
    }
}


In [ ]:
import "dotenv/config";
import { generateText } from "ai";

// Trả về 1 cục text
const result = await generateText({
    model: "anthropic/claude-haiku-4.5",
    prompt: "Tôi tên là gì?",
});

console.log(result);

In [ ]:
// ES6+
import "dotenv/config";
import { streamText } from "ai";

// Result chưa phải là kết quả cuối cùng
const result = streamText({
    model: "anthropic/claude-haiku-4.5",
    prompt: "Bạn có thể giúp tôi những gì?",
});

// Lặp và nhả về từng từ 1
for await (const textPart of result.textStream) {
    process.stdout.write(textPart); // In ra Terminal
    // Web Socket: Pusher bắn lên
    // pusher.trigger("channel", "event-name", {data: "123"})
    // FE subscribe => lấy dữ liệu -> setState -> render ra giao diện
}

console.log();
console.log("Token usage:", await result.usage);
console.log("Finish reason:", await result.finishReason);


In [ ]:
// Generate Image
// ES6+
import "dotenv/config";
import { generateText } from "ai";
import fs from "fs";

const result = await generateText({
    model: "google/gemini-3-pro-image",
    prompt: "Hãy tạo cho tôi một bức ảnh phong cảnh đẹp",
});

// Nano Banana models return images in result.files with uint8Array
const imageFiles = result.files.filter((f) =>
    f.mediaType?.startsWith("image/"),
);

if (imageFiles.length > 0) {
    // Lấy ra đuôi của file ".png"
    const extension = imageFiles[0].mediaType?.split("/")[1] || "png";
    fs.writeFileSync(
        `./public/images/output.${extension}`,
        imageFiles[0].uint8Array,
    );
    console.log(`Image saved to output.${extension}`);
}


In [ ]:
// Search 
// ES6+
import "dotenv/config";
import { gateway, streamText } from "ai";

const result = streamText({
    model: "anthropic/claude-haiku-4.5", // Works with any model, not just Perplexity
    prompt: "Tìm cho tôi 5 kết quả về sự việc Mỹ và Iran",
    maxSteps: 5,
    tools: {
        perplexity_search: gateway.tools.perplexitySearch({
            maxResults: 5, // Trả  ra bao nhiêu kết quả
            country: "VN", // Tìm trong nước
            searchLanguageFilter: ["vi"], // Ngôn ngữ VN
            searchRecencyFilter: "day", // Search gần đây
        }),
    },
});

for await (const part of result.fullStream) {
    if (part.type === "text-delta") {
        process.stdout.write(part.text);
    } else if (part.type === "tool-call") {
        console.log("Tool call:", part.toolName);
    } else if (part.type === "tool-result") {
        console.log("Search results received");
    }
}
